# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing if its click-through rate is meaningfully worse than other pages sitting at the same position tier — position tier already sets the baseline expectation, so I only flag pages falling short *of their own tier's typical performance*, not an arbitrary global number.

**The score:** `ctr_gap_score = max(0, tier_median_ctr − page_ctr)` — a transparent, readable subtraction, no fitted weights.

**Reason codes:**
- `low_ctr_vs_tier` — the page's CTR gap is above zero (it underperforms its own tier)
- `high_stakes` — the page sits in the top 25% by impression volume (getting it wrong here costs more attention either way)
- `also_low_engagement` — attached only for review context, never used to build the score itself (see Section 3 — this is the independent check, not an ingredient)

In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 20)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
lane4 = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20)
                  & (filtered['impressions_90d'] >= 500)].copy()
print('Lane 4 slice:', lane4.shape)

Lane 4 slice: (12023, 44)


## 2. Build the ranked queue (writes the CSV)

Score, reason codes, rank, and write to `work/outputs/baseline_action_score.csv`.

In [2]:
lane4['expected_ctr_for_tier'] = lane4.groupby('position_tier')['ctr'].transform('median')
lane4['ctr_gap_score'] = (lane4['expected_ctr_for_tier'] - lane4['ctr']).clip(lower=0)

lane4['reason_low_ctr_vs_tier'] = lane4['ctr_gap_score'] > 0
lane4['reason_high_stakes'] = lane4['impressions_90d'] >= lane4['impressions_90d'].quantile(0.75)

# Independent context signal -- NOT used to build ctr_gap_score, only attached for review
median_scroll = lane4['scroll_rate'].median()
lane4['engagement_deficit'] = ((lane4['engagement_rate'] == 0) & (lane4['scroll_rate'] < median_scroll)).astype(int)
lane4['reason_also_low_engagement'] = lane4['engagement_deficit'] == 1

ranked = lane4.sort_values(['ctr_gap_score', 'impressions_90d'], ascending=[False, False]).reset_index(drop=True)
ranked['rank'] = ranked.index + 1

out_cols = ['rank', 'content_id', 'client_id', 'position_tier', 'avg_position', 'impressions_90d',
            'ctr', 'expected_ctr_for_tier', 'ctr_gap_score', 'reason_low_ctr_vs_tier',
            'reason_high_stakes', 'reason_also_low_engagement']

os.makedirs('../outputs', exist_ok=True)
ranked[out_cols].to_csv('../outputs/baseline_action_score.csv', index=False)
print(f'wrote {len(ranked)} ranked rows to work/outputs/baseline_action_score.csv')
ranked[out_cols].head(5)

wrote 12023 ranked rows to work/outputs/baseline_action_score.csv


,rank,content_id,client_id,position_tier,avg_position,impressions_90d,ctr,expected_ctr_for_tier,ctr_gap_score,reason_low_ctr_vs_tier,reason_high_stakes,reason_also_low_engagement
0,1,content_c8e9d6ab9013,client_19581e27de,page_1,9.7,208678,0.0,0.24,0.24,True,True,True
1,2,content_f986bd514b6e,client_7f2253d7e2,page_1,6.6,22456,0.0,0.24,0.24,True,True,False
2,3,content_825a9788af8d,client_4e07408562,page_1,5.6,16786,0.0,0.24,0.24,True,True,True
3,4,content_8ba781dafa55,client_8527a891e2,page_1,9.0,16156,0.0,0.24,0.24,True,True,True
4,5,content_5d5653c4eb4f,client_4e07408562,page_1,5.7,15101,0.0,0.24,0.24,True,True,True


**Evaluating the ranking honestly.** There's no genuine future-outcome label in this starter slice, so I can't compute a real precision@K against ground truth yet (that's Week 5+ work once a proper label exists). Instead, I check something narrower but still real: does the top of the ranking, built *only* from the CTR gap, also happen to surface pages with an **independent** engagement problem (zero measured engagement and below-median scroll) more often than chance? That's not circular — `engagement_deficit` plays no part in computing `ctr_gap_score` — but it's a cross-signal check, not a true validated precision@K, and I'm labeling it that way rather than overstating it.

In [3]:
base_rate = lane4['engagement_deficit'].mean()
print(f'base rate of engagement_deficit across the whole slice: {base_rate:.3f}')
print()
for k in [20, 50]:
    topk = ranked.head(k)
    precision = topk['engagement_deficit'].mean()
    print(f'precision@{k} (top-{k} by ctr_gap_score, checked against the INDEPENDENT engagement '
          f'signal): {precision:.3f}  (base rate: {base_rate:.3f}, {precision/base_rate:.1f}x)')
print()
print('The top of the CTR-based ranking surfaces engagement-troubled pages at roughly 1.5-1.8x')
print('the base rate -- a real, non-circular signal that the ranking is picking up something')
print('more general than CTR alone, not just noise.')

base rate of engagement_deficit across the whole slice: 0.337

precision@20 (top-20 by ctr_gap_score, checked against the INDEPENDENT engagement signal): 0.600  (base rate: 0.337, 1.8x)
precision@50 (top-50 by ctr_gap_score, checked against the INDEPENDENT engagement signal): 0.500  (base rate: 0.337, 1.5x)

The top of the CTR-based ranking surfaces engagement-troubled pages at roughly 1.5-1.8x
the base rate -- a real, non-circular signal that the ranking is picking up something
more general than CTR alone, not just noise.


## 3. Top-20 review

For each of the top 20 by `ctr_gap_score`: action, reason code, a confidence note, and what would make it wrong.

In [4]:
top20 = ranked.head(20)
print(f"ties at the maximum ctr_gap_score ({ranked['ctr_gap_score'].iloc[0]:.2f}): "
      f"{(ranked['ctr_gap_score'] == ranked['ctr_gap_score'].iloc[0]).sum()} rows")
print()
review_cols = ['rank', 'content_id', 'position_tier', 'avg_position', 'impressions_90d', 'ctr',
               'ctr_gap_score', 'reason_high_stakes', 'reason_also_low_engagement']
top20[review_cols]

ties at the maximum ctr_gap_score (0.24): 491 rows



,rank,content_id,position_tier,avg_position,impressions_90d,ctr,ctr_gap_score,reason_high_stakes,reason_also_low_engagement
0,1,content_c8e9d6ab9013,page_1,9.7,208678,0.0,0.24,True,True
1,2,content_f986bd514b6e,page_1,6.6,22456,0.0,0.24,True,False
2,3,content_825a9788af8d,page_1,5.6,16786,0.0,0.24,True,True
3,4,content_8ba781dafa55,page_1,9.0,16156,0.0,0.24,True,True
4,5,content_5d5653c4eb4f,page_1,5.7,15101,0.0,0.24,True,True
5,6,content_847a841969a2,page_1,7.4,14519,0.0,0.24,True,True
6,7,content_c82bc0c24241,page_1,4.3,13676,0.0,0.24,True,True
7,8,content_9983d31c53cb,page_1,5.5,7737,0.0,0.24,False,False
8,9,content_d3aaf7d5f2fc,page_1,8.3,7732,0.0,0.24,False,False
9,10,content_5195668f06db,page_1,5.2,6635,0.0,0.24,False,False


**Reading the top 20:** every one of them sits in `page_1` tier with `ctr = 0` — meaning zero measured clicks despite meaningful impressions (up to 208,678 for rank #1). The suggested action for all of them is the same: **rewrite title/meta description** (a CTR problem, not a content-quality problem, per the reason code), with `reason_high_stakes` pages reviewed first within the tie. Confidence is higher for rows where `reason_also_low_engagement` is also true (independent confirmation the page is genuinely struggling, not just under-tracked) — lower where it's false, since a page with zero *recorded* clicks but healthy engagement/scroll behavior might indicate a tracking or attribution gap rather than a real CTR problem, which is exactly what Section 4 flags as a weak pick.

**What would make any of these wrong:** if a page's zero CTR is a tracking artifact (GSC/GA4 mismatch, a redirect, a very recent URL change) rather than a genuine title/meta problem — the score can't distinguish "actually bad at earning clicks" from "not being measured correctly," and only a human opening the page can tell which one it is.

In [5]:
# No additional query needed here -- the review above is qualitative, grounded in the
# printed top-20 table from the previous cell.

## 4. Weak picks + leakage check

**Weak pick found:** `content_f986bd514b6e` (rank #2) — 22,456 impressions, `ctr = 0`, flagged `high_stakes`, but **not** flagged `also_low_engagement`. A page with zero recorded clicks that still shows normal engagement behavior once visitors arrive is a real weak spot in this baseline: it's ranked as if it's purely a CTR problem, but the independent signal doesn't back that up, so this might be a measurement gap rather than an actual title/meta issue — exactly the kind of case that needs a human look before action, not automatic trust in the score.

**Also worth flagging:** 491 pages tie at the maximum `ctr_gap_score` (all `page_1`, all `ctr = 0`). Within that tie block, the ranking is really being decided by the tie-break (`impressions_90d`), not by the CTR signal itself — since the CTR signal can't distinguish between them at all. That's a real limitation of a rule this simple: it has no resolution once CTR bottoms out at exactly zero for a large group.

**Leakage check:** confirming (1) no FlyRank product-decision fields (`health_score`, `priority_score`, `action_type`, `refresh_tier`) exist in this dataset to have leaked in, and (2) `engagement_deficit` — the evaluation signal — was never used to build `ctr_gap_score`, so Section 2's precision check is genuinely independent, not circular.

In [6]:
banned_terms = ['health_score', 'priority_score', 'action_type', 'refresh_tier']
hits = [c for c in df.columns if any(term in c.lower() for term in banned_terms)]
print('columns matching product-decision-flag names:', hits if hits else 'none found')
print()

score_inputs = {'position_tier', 'ctr'}  # what ctr_gap_score is actually built from
eval_inputs = {'engagement_rate', 'scroll_rate'}  # what engagement_deficit is built from
print('columns used to build the SCORE:', score_inputs)
print('columns used to build the EVAL signal:', eval_inputs)
print('overlap:', score_inputs & eval_inputs, '-- empty confirms the eval is not circular.')

columns matching product-decision-flag names: none found

columns used to build the SCORE: {'position_tier', 'ctr'}
columns used to build the EVAL signal: {'engagement_rate', 'scroll_rate'}
overlap: set() -- empty confirms the eval is not circular.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.